# 🩺 Diabetes Classification — Random Forest + GridSearchCV

![Python](https://img.shields.io/badge/Python-3.8+-blue?logo=python)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.0+-orange?logo=scikit-learn)
![Task](https://img.shields.io/badge/Task-Classification-green)
![Dataset](https://img.shields.io/badge/Dataset-Pima%20Indians%20Diabetes-lightgrey)

---

## 📌 Project Overview

This project builds a **binary classification model** to predict whether a patient has **diabetes** based on medical diagnostic measurements.

| Item | Detail |
|------|--------|
| **Dataset** | Pima Indians Diabetes Dataset |
| **Model** | Random Forest + GridSearchCV |
| **Metric** | Precision |
| **Target** | `Outcome` → 1 = Diabetes, 0 = Healthy |

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print('✅ Libraries loaded successfully!')

## 📂 2. Load & Explore Data

> 📥 Dataset: [Pima Indians Diabetes — Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
>
> Download `diabetes.csv` or `diabetes.xlsx` and place it in the same folder as this notebook.

In [ ]:
import os

# Auto-detect file format
if os.path.exists('diabetes.csv'):
    data = pd.read_csv('diabetes.csv')
    print('✅ Loaded diabetes.csv')
elif os.path.exists('diabetes.xlsx'):
    data = pd.read_excel('diabetes.xlsx')
    print('✅ Loaded diabetes.xlsx')
else:
    raise FileNotFoundError('❌ Please place diabetes.csv or diabetes.xlsx in this folder!')

print(f'Shape: {data.shape}')
data.head()

In [ ]:
# Basic info
data.info()

In [ ]:
# Statistical summary
data.describe()

In [ ]:
# Class distribution
print('Class Distribution:')
print(data['Outcome'].value_counts())

plt.figure(figsize=(5, 4))
data['Outcome'].value_counts().plot(
    kind='bar', color=['steelblue', 'tomato'], edgecolor='black'
)
plt.xticks([0, 1], ['Healthy (0)', 'Diabetes (1)'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(data.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## ✂️ 3. Split Data — Train & Test

In [ ]:
target = 'Outcome'
X = data.drop(target, axis=1)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size: {X_train.shape[0]} samples')
print(f'Test size : {X_test.shape[0]} samples')

## ⚙️ 4. Preprocessing — Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('✅ Features scaled with StandardScaler')

## 🔍 5. Hyperparameter Tuning — GridSearchCV

In [ ]:
params = {
    'n_estimators'     : [50, 100, 200, 500],
    'criterion'        : ['gini', 'entropy', 'log_loss'],
    'max_depth'        : [None, 2, 5],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=100),
    param_grid=params,
    scoring='precision',
    cv=6,
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(f'\n✅ Best CV Precision : {grid.best_score_:.4f}')
print(f'✅ Best Parameters   : {grid.best_params_}')

## 📊 6. Evaluate Model

In [ ]:
y_pred = grid.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Diabetes']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Healthy', 'Actual: Diabetes'],
    columns=['Pred: Healthy', 'Pred: Diabetes']
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix — Diabetes Classification')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
best_model    = grid.best_estimator_
feat_importance = pd.Series(
    best_model.feature_importances_,
    index=data.drop(target, axis=1).columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_importance.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 🏁 7. Summary

| Item | Detail |
|------|--------|
| **Model** | Random Forest Classifier |
| **Tuning** | GridSearchCV (6-fold Cross Validation) |
| **Metric Optimized** | Precision |
| **Train / Test Split** | 80% / 20% |
| **Scaling** | StandardScaler |

### 🔑 Key Takeaways
- ✅ **Glucose** is the most important feature for predicting diabetes
- ✅ **BMI** and **Age** are the next most influential features
- ✅ Precision-optimized model reduces false positives — critical in medical diagnosis
- ✅ GridSearchCV automatically finds the best hyperparameter combination